# Installing Spotipy

In [ ]:
##!conda install -c conda-forge spotipy

## Loading credentials from the config file

#### Make sure that you have stored your spotify client_id and client_secret in a separate config.py file
#### Once that's done, we import it

In [7]:
from auth import credentials
import json
import pandas as pd
import pprint
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

credentials = credentials()
sp = spotipy.Spotify(auth_manager=credentials)

## Starting with Spotify API

In [34]:
gaga = sp.search(q = "lady gaga", limit=None)

In [35]:
for item in gaga["tracks"]["items"]:
    print(item["name"])

Abracadabra
Shallow
Bloody Mary
Always Remember Us This Way
Poker Face
Abracadabra
Judas
Bad Romance
Born This Way
Just Dance
The Edge Of Glory
Hold My Hand
Million Reasons
Paparazzi
I'll Never Love Again - Film Version
Hold My Hand


# Understanding the json

## Other Info

In [36]:
results = sp.search(q='Lady Gaga', limit=50)

In [38]:
results['tracks']['items'][0]['artists']

[{'external_urls': {'spotify': 'https://open.spotify.com/artist/1HY2Jd0NmPuamShAr6KMms'},
  'href': 'https://api.spotify.com/v1/artists/1HY2Jd0NmPuamShAr6KMms',
  'id': '1HY2Jd0NmPuamShAr6KMms',
  'name': 'Lady Gaga',
  'type': 'artist',
  'uri': 'spotify:artist:1HY2Jd0NmPuamShAr6KMms'}]

In [39]:
results.keys() # We can see that we only have tracks

dict_keys(['tracks'])

In [40]:
results["tracks"].keys() # Let's check the values

dict_keys(['href', 'limit', 'next', 'offset', 'previous', 'total', 'items'])

In [41]:
results["tracks"]["href"] # Query we have searched 

'https://api.spotify.com/v1/search?offset=0&limit=50&query=Lady%20Gaga&type=track'

In [42]:
results["tracks"]["items"] #items (actual tracks)

[{'album': {'album_type': 'single',
   'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/1HY2Jd0NmPuamShAr6KMms'},
     'href': 'https://api.spotify.com/v1/artists/1HY2Jd0NmPuamShAr6KMms',
     'id': '1HY2Jd0NmPuamShAr6KMms',
     'name': 'Lady Gaga',
     'type': 'artist',
     'uri': 'spotify:artist:1HY2Jd0NmPuamShAr6KMms'}],
   'available_markets': ['AR',
    'AU',
    'AT',
    'BE',
    'BO',
    'BR',
    'BG',
    'CA',
    'CL',
    'CO',
    'CR',
    'CY',
    'CZ',
    'DK',
    'DO',
    'DE',
    'EC',
    'EE',
    'SV',
    'FI',
    'FR',
    'GR',
    'GT',
    'HN',
    'HK',
    'HU',
    'IS',
    'IE',
    'IT',
    'LV',
    'LT',
    'LU',
    'MY',
    'MT',
    'MX',
    'NL',
    'NZ',
    'NI',
    'NO',
    'PA',
    'PY',
    'PE',
    'PH',
    'PL',
    'PT',
    'SG',
    'SK',
    'ES',
    'CH',
    'TW',
    'TR',
    'UY',
    'US',
    'GB',
    'AD',
    'LI',
    'MC',
    'ID',
    'JP',
    'TH',
    'VN',
    'RO',
    

In [43]:
results["tracks"]["limit"]#Limit we have chosen

50

In [44]:
results["tracks"]["next"] #link to the next page (next 50 tracks)

'https://api.spotify.com/v1/search?offset=50&limit=50&query=Lady%20Gaga&type=track'

In [45]:
results["tracks"]["offset"] # Actual offset (starting point)

0

In [46]:
results["tracks"]["previous"] #Previous search

In [47]:
results["tracks"]["total"] # Number of matches

1000

## Getting the track_id

In [55]:
results["tracks"]["items"][0]["id"]

'5ZLUm9eab8y3tqQ1OhQSHI'

## Embedded track player

In [48]:
from IPython.display import IFrame

In [56]:
def get_playlist_tracks(username, playlist_id):
    results = sp.user_playlist_tracks(username,playlist_id,market="GB")
    tracks = results['items']
    while results['next']:
        results = sp.next(results)
        tracks.extend(results['items'])
    return tracks

In [76]:
def play_song(track_id):
    return IFrame(src="https://open.spotify.com/embed/track/"+track_id,
       width="320",
       height="80",
       frameborder="0",
       allowtransparency="true",
       allow="encrypted-media",)

In [77]:
play_song("3y1vPgwHA2R20efomVmD9F")

# Getting the Audio features of a song

## Building a Data frame of audio features

In [ ]:
# this doesn't work anymore in spotipy

## Searching a playlist

In [99]:
chopin = sp.user_playlist_tracks(user=None, playlist_id="5I6Pb8tVHfx3su0RRiMqYF", limit=100)

## Extracting a song from playlist

In [109]:
for song in chopin['items']:
    print(song['track']['name'])

2 Waltzes, Op. posth. 69: No. 2 in B Minor
Waltz No. 10 in B Minor, Op. 69 No. 2
Waltz in B Minor, Op. 69 No. 2 (Ed. J. Fontana)
Waltz No. 10 in B Minor, Op. 69 No. 2
Waltz No. 10 in B Minor, Op. 69 No. 2


## Extracting the songs of a playlist

Pagination using "next".
When you collect songs from a playlist using sp.playlist_tracks, you're limited by the limit parameter, which has a maximum (and default) value of 100. When the playlist has more than 100 songs, you have to collect them by navigating through the "pages" of the results.

The parameter offset allows you to retrieve resuls starting at a certain position: if you start at position 101, you'd get the next "page" of results. An offset of 201 would give you the third page, and so on.

The function sp.next() does the same, but in a simpler way: it can be used on the results from any request to directly retrieve the results for the next page.

We can check whether there's a next page or not by accessing the key next on the results from any request.

In [155]:
chopin.next()

AttributeError: 'dict' object has no attribute 'next'

### Chopping a big playlist into chunks

In [137]:
flamenco = sp.playlist("4ZfGmEZcdbchpmikea6vNt")

In [156]:
for song in flamenco['tracks']['items']:
    print(song['track']['name'])

Lo Que Tu Querer Me Cuesta - Soleá
Romance de la Plata - Versión Flamenco
El Emigrante
Dulce Tiranía
No Solo Yo (Ambient Mix)
La Sangre de Mi Corazón
Anoche Me Preguntabas
Dama Errante
Tonto
Nochecita Sanjuanera
Eso Nunca lo Diré (Granaína)
Vampiro
Solos Tú y Yo (Tangos)
Telarañas. #GitanasEnEstereo
Están Bailando (Sevillanas)
Malditos benditos, benditos malditos (feat. Bunbury)
Arrímate (Fandangos)
Oración
Yo Escucho los Cantos
Malagueña de la Trini
Esta No Es Manera de Decir Adiós
Andaluces de Jaén
Con Toda Palabra
Tangos de la Repompa
Caricias de Sal
Si Tú Supieras Compañero
Bulerías Cortas
Morentangos
Banderas Republicanas
Las Manillas Del Reloj - Tangos
El Venadito - Colombiana
Por Tu Querer Como un Niño (Fandango)
Por Qué Será
La Misa Que Voy Yo (Soleá con Auto-tune)
La Alondra
Grandes Locuras (Alegrías)
Aunque Es De Noche
Catalina
La Ciudad de los Gitanos
Todavía
Tú Que Vienes a Rondarme
Lola
Tiene La Tarara
90 Minutos
Camino De Tu Corazón - Rumba
A Luis De La Pica - Bulerías
Lá

# Getting albums 

In this section we will work with albums to extract information. We will start by extracting all the albums of an artist.

In [114]:
coldplay_code = "4gzpq5DPGxSnKTe4SA8HAU"
results = sp.artist_albums(coldplay_code, album_type='album')

In [121]:
for album in results['items']:
    print(album['name'])

Moon Music
Moon Music (Full Moon Edition)
Music Of The Spheres
Everyday Life
Live in Buenos Aires
A Head Full of Dreams
Ghost Stories Live 2014
Ghost Stories
Live 2012
Mylo Xyloto
LeftRightLeftRightLeft (Live)
Viva La Vida (Prospekt's March Edition)
Viva La Vida or Death and All His Friends
X&Y
Live 2003
A Rush of Blood to the Head
Parachutes


## Getting the songs of a given album

In [122]:
parachutes = "6ZG5lRT77aJ3btmArcykra"
results = sp.album(parachutes)

In [136]:
for track in results['tracks']['items']:
    print(track['name'])

Don't Panic
Shiver
Spies
Sparks
Yellow
Trouble
Parachutes
High Speed
We Never Change
Everything's Not Lost - Includes Hidden Track 'Life Is For Living'
